# Setup partagé — Cardinal Buoy Classification
Exécuter via `%run setup.ipynb` depuis chaque notebook.
Produit : `dataset`, `X` (uint8 crops), `y` (labels int), et toutes les constantes communes.

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

In [ ]:
# Direction mappings — identiques dans tous les notebooks
DIRECTION_MAPPING = {'E': 0, 'N': 1, 'S': 2, 'W': 3}
DIRECTION_NAMES   = {0: 'East', 1: 'North', 2: 'South', 3: 'West'}
DIRECTION_DIRS    = {
    'E': ('E', 'East'), 'N': ('N', 'North'),
    'S': ('S', 'South'), 'W': ('W', 'West')
}

# Alias utilisés dans certains notebooks
direction_mapping = DIRECTION_MAPPING
direction_names   = DIRECTION_NAMES
reverse_map       = DIRECTION_NAMES

# Constantes image
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.gif', '.bmp')
image_extensions = IMAGE_EXTENSIONS

# Constantes couleur HSV — bouées cardinales jaune et noire
# Jaune : teinte 8–38° (permet un léger orange-jaune), saturation et valeur élevées
YELLOW_LOWER_HSV = np.array([8,  50,  60],  dtype=np.uint8)
YELLOW_UPPER_HSV = np.array([38, 255, 255], dtype=np.uint8)
# Noir : faible luminosité ET faible saturation (exclut l'eau sombre bleue)
BLACK_V_MAX      = 55   # seuil sur la valeur HSV
BLACK_S_MAX      = 110  # seuil sur la saturation (eau = saturation > 110)
BLACK_THRESHOLD  = BLACK_V_MAX  # alias rétrocompatible

# Cross-validation
CV_FOLDS = 5

# Méthode de détection de la bouée dans load_dataset()
# 'algo'        : détection couleur + morphologie (detect_buoy)
# 'annotations' : bounding boxes YOLO annotées dans Project datasets/N/, E/, S/, W/
DETECTION_METHOD = 'algo'

# Trouve le dossier 'Project datasets' depuis le CWD ou son parent
_cwd = Path.cwd()
if (_cwd / 'Project datasets').exists():
    dataset_root = _cwd / 'Project datasets'
elif (_cwd.parent / 'Project datasets').exists():
    dataset_root = _cwd.parent / 'Project datasets'
else:
    raise FileNotFoundError(f"'Project datasets' introuvable depuis {_cwd}")

print(f"dataset_root : {dataset_root}")
print(f"DETECTION_METHOD : {DETECTION_METHOD}")

In [ ]:
def _iou(a, b):
    """Intersection over Union entre deux boîtes (xmin, ymin, xmax, ymax)."""
    xi1, yi1 = max(a[0], b[0]), max(a[1], b[1])
    xi2, yi2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _nms(candidates, iou_thresh=0.4):
    """Non-Maximum Suppression. candidates = [(score, xmin, ymin, xmax, ymax)]."""
    remaining = sorted(candidates, key=lambda c: c[0], reverse=True)
    kept = []
    while remaining:
        best = remaining.pop(0)
        kept.append(best)
        remaining = [c for c in remaining if _iou(best[1:], c[1:]) < iou_thresh]
    return kept


def _make_masks(hsv):
    """
    Retourne (yellow_mask, black_mask) binaires uint8 (0 / 255).

    Jaune : plage HSV large pour couvrir les variantes d'éclairage.
    Noir  : faible valeur ET faible saturation pour exclure l'eau sombre
            qui est typiquement bleue (saturation > BLACK_S_MAX).
    """
    yellow = cv2.inRange(hsv, YELLOW_LOWER_HSV, YELLOW_UPPER_HSV)
    low_v  = (hsv[:, :, 2] < BLACK_V_MAX).astype(np.uint8)
    low_s  = (hsv[:, :, 1] < BLACK_S_MAX).astype(np.uint8)
    black  = (low_v & low_s).astype(np.uint8) * 255
    return yellow, black


def _score_window(hsv_window):
    """
    Score d'une fenêtre pour la présence d'une bouée cardinale.

    Les bouées cardinales sont JAUNES et NOIRES : les deux couleurs doivent
    être présentes. Un bonus est accordé si les proportions sont équilibrées.
    Retourne 0 si l'une des couleurs est absente.
    """
    yellow, black = _make_masks(hsv_window)
    total    = hsv_window.shape[0] * hsv_window.shape[1]
    n_yellow = np.count_nonzero(yellow)
    n_black  = np.count_nonzero(black)

    if n_yellow == 0 or n_black == 0:
        return 0.0

    buoy_ratio = (n_yellow + n_black) / total
    # Bonus si les deux couleurs sont bien équilibrées (idéalement ~50/50)
    balance = min(n_yellow, n_black) / max(n_yellow, n_black)
    return buoy_ratio * (0.6 + 0.4 * balance)


def detect_buoy(image_rgb,
                scales=(0.15, 0.25, 0.40, 0.60),
                aspect_ratio=0.45,
                stride=12,
                min_score=0.08,
                iou_thresh=0.35):
    """
    Localise la bouée cardinale (jaune/noire) dans une image RGB.

    Stratégie
    ---------
    1. Morphologie : masques jaune + noir → fermeture pour relier les bandes
       → contours → score de chaque candidat région.
    2. Sliding window multi-échelle (fallback si l'étape 1 échoue).

    Retourne (xmin, ymin, xmax, ymax).
    """
    h, w = image_rgb.shape[:2]
    hsv  = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    yellow_mask, black_mask = _make_masks(hsv)

    # ── 1. Approche morphologique ──────────────────────────────────────────
    buoy_mask = cv2.bitwise_or(yellow_mask, black_mask)

    # Fermeture : relie les bandes jaune et noire d'une même bouée.
    # Taille du noyau ≈ 4 % de la hauteur image (quelques dizaines de pixels).
    ksize  = max(7, h // 25)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (ksize, ksize))
    closed = cv2.morphologyEx(buoy_mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    morph_best = None
    morph_best_score = -1.0
    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cw * ch
        # Ignorer les régions trop petites ou trop grandes
        if area < 0.005 * h * w or area > 0.85 * h * w:
            continue
        score = _score_window(hsv[y:y+ch, x:x+cw])
        if score > morph_best_score:
            morph_best_score = score
            morph_best = (x, y, x + cw, y + ch)

    if morph_best is not None and morph_best_score >= min_score:
        return morph_best

    # ── 2. Sliding window fallback ─────────────────────────────────────────
    candidates = []
    for scale in scales:
        win_h = int(h * scale)
        win_w = int(win_h * aspect_ratio)
        if win_h < 16 or win_w < 16:
            continue
        for y in range(0, h - win_h + 1, stride):
            for x in range(0, w - win_w + 1, stride):
                score = _score_window(hsv[y:y+win_h, x:x+win_w])
                if score >= min_score:
                    candidates.append((score, x, y, x + win_w, y + win_h))

    if not candidates:
        return 0, 0, w, h  # fallback total : image entière

    _, xmin, ymin, xmax, ymax = _nms(candidates, iou_thresh)[0]
    return xmin, ymin, xmax, ymax


def load_annotation_bbox(img_file, letter):
    """
    Charge la bounding box annotée YOLO pour une image donnée.
    Le fichier .txt est cherché dans dataset_root / letter / stem.txt
    (ex. dataset_root / 'N' / 'N_001.txt').

    Format YOLO : class_id cx_norm cy_norm w_norm h_norm (valeurs normalisées [0,1])

    Retourne (xmin, ymin, xmax, ymax) en pixels, ou None si le fichier est absent.
    """
    label_path = dataset_root / letter / (Path(img_file).stem + '.txt')
    if not label_path.exists():
        return None
    parts = label_path.read_text().strip().split('\n')[0].split()
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = (float(v) for v in parts)
    img = cv2.imread(str(img_file))
    if img is None:
        return None
    h, w = img.shape[:2]
    xmin = max(0, int((cx - bw / 2) * w))
    ymin = max(0, int((cy - bh / 2) * h))
    xmax = min(w, int((cx + bw / 2) * w))
    ymax = min(h, int((cy + bh / 2) * h))
    return xmin, ymin, xmax, ymax

In [ ]:
def load_dataset(img_size=(128, 128)):
    """
    Charge les bouées cardinales en détectant la région automatiquement.

    La méthode de détection dépend de DETECTION_METHOD :
      'annotations' : lit les fichiers .txt YOLO dans dataset_root / letter /
      'algo'        : sliding window multi-échelle (detect_buoy)

    Retourne :
      dataset : dict {direction_id (int): [{'cropped': np.ndarray uint8}, ...]}
      X       : np.ndarray (N, H, W, 3) uint8
      y       : np.ndarray (N,) int
    """
    dataset_local = {}
    images, labels = [], []

    for letter, direction_id in DIRECTION_MAPPING.items():
        img_dir = dataset_root / DIRECTION_DIRS[letter][1]
        dataset_local[direction_id] = []

        img_files = sorted([
            p for ext in IMAGE_EXTENSIONS
            for p in img_dir.glob(f'*{ext}')
        ])
        print(f"{DIRECTION_NAMES[direction_id]}: {len(img_files)} images")

        for img_file in img_files:
            try:
                image = cv2.imread(str(img_file))
                if image is None:
                    continue
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                if DETECTION_METHOD == 'annotations':
                    bbox = load_annotation_bbox(img_file, letter)
                    if bbox is None:
                        # fallback sur l'algo si l'annotation est manquante
                        xmin, ymin, xmax, ymax = detect_buoy(image)
                    else:
                        xmin, ymin, xmax, ymax = bbox
                else:
                    xmin, ymin, xmax, ymax = detect_buoy(image)

                cropped = image[ymin:ymax, xmin:xmax]
                if cropped.size == 0:
                    continue

                cropped = cv2.resize(cropped, img_size)
                dataset_local[direction_id].append({'cropped': cropped})
                images.append(cropped)
                labels.append(direction_id)
            except Exception as e:
                print(f"  Erreur {img_file.name}: {e}")
                continue

    X_out = np.array(images, dtype=np.uint8)
    y_out = np.array(labels,  dtype=np.int32)
    return dataset_local, X_out, y_out

In [ ]:
# IMG_SIZE peut être défini AVANT %run pour le surcharger (ex. ResNet : 224x224)
if 'IMG_SIZE' not in dir():
    IMG_SIZE = (128, 128)

FIXED_SIZE = IMG_SIZE  # alias utilisé dans les notebooks ML

# LOAD_DATA peut être mis à False avant %run pour ne pas charger les images
# (utile pour les notebooks classiques qui n'utilisent pas les crops)
_load = globals().pop('LOAD_DATA', True)
if _load:
    dataset, X, y = load_dataset(IMG_SIZE)
    print(f"Dataset chargé : {len(y)} images, taille {IMG_SIZE}")
    print(f"Distribution   : {dict(zip(DIRECTION_NAMES.values(), np.bincount(y)))}")
else:
    print(f"Setup chargé — dataset_root : {dataset_root}")